# OCR Prior — Recall / Coverage Check (DocTamper)

Verifies the **OCR text prior** *before* any training — the TA's headline ask. The OCR mask is a
region-proposal stage, so we measure how well it **covers the tampered regions** (recall). A missed
forged region cannot be recovered downstream.

**Safety:** this notebook does **not** load any DocTamper checkpoint (`.pth`) or pickle (`.pk`). It only
reads `(image, tamper_mask)` pairs from the LMDB (using DocTamper's documented key format) and runs a
*pretrained* OCR detector. No `torch.load` / `pickle.load` of upstream files.

**Flow:** clone our repo → **unzip** the DocTamperV1-FCD archive you downloaded to Drive (password from
the author's email) → read LMDB → build `M_text` with pretrained EasyOCR → `OCRCoverageMeter`
→ qualitative overlays.

> You download the official zip yourself (from the author's Drive link) and put it on your Drive; this
> notebook only unzips it.

In [ ]:
# --- Setup: mount Drive, clone our repo, install lightweight deps ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys
%cd /content
if not os.path.exists('/content/HLCV-Project'):
    !git clone https://github.com/SamiraAbedini/HLCV-Project.git

# Only what the OCR-recall check needs. DocTamper code itself is NOT cloned or run here.
!pip -q install lmdb easyocr opencv-python-headless pillow six tqdm

sys.path.append('/content/HLCV-Project')
print('setup done')

In [ ]:
# --- Unzip the DocTamper archive you downloaded to Drive ---
# You download the official zip yourself and place it on your Drive; this cell
# unzips it with the password (typed via getpass, never written to the notebook).
import os, glob, getpass

ZIP_PATH = '/content/drive/MyDrive/DocTamperV1-FCD.zip'  # <-- set to where YOUR zip is
WORK     = '/content/data'                               # extracted LMDB target (this session)
os.makedirs(WORK, exist_ok=True)

assert os.path.exists(ZIP_PATH), f'Archive not found: {ZIP_PATH} (edit ZIP_PATH to your zip).'

# Unzip only if not already extracted this session.
if not glob.glob(os.path.join(WORK, '**', 'data.mdb'), recursive=True):
    pw = getpass.getpass('DocTamper zip password: ')   # typed now, not stored
    !unzip -P "$pw" "{ZIP_PATH}" -d "{WORK}"
    del pw

# Locate the LMDB folder (the dir containing data.mdb), regardless of nesting.
mdbs = glob.glob(os.path.join(WORK, '**', 'data.mdb'), recursive=True)
assert mdbs, 'Unzip did not produce a data.mdb — check the password and the archive.'
DATA_DIR = os.path.dirname(mdbs[0])
print('LMDB folder ready:', DATA_DIR)

In [ ]:
# --- Config + LMDB reader (DocTamper key format: image-%09d / label-%09d) ---
# Uses DATA_DIR set by the download cell above.
import io, lmdb, cv2, numpy as np
from PIL import Image

N_SAMPLES = 200   # images to evaluate; a few hundred gives a stable estimate

assert 'DATA_DIR' in globals() and os.path.exists(DATA_DIR), 'Run the download cell first.'
print('using DATA_DIR =', DATA_DIR)

env = lmdb.open(DATA_DIR, readonly=True, lock=False, readahead=False, meminit=False)
with env.begin(write=False) as txn:
    num_samples = int(txn.get(b'num-samples'))
print('num-samples in dataset:', num_samples)

def read_sample(index):
    """Return (rgb_uint8 HxWx3, tamper_mask_bool HxW)."""
    with env.begin(write=False) as txn:
        imgbuf = txn.get(('image-%09d' % index).encode('utf-8'))
        lblbuf = txn.get(('label-%09d' % index).encode('utf-8'))
    image = np.array(Image.open(io.BytesIO(imgbuf)).convert('RGB'))
    mask = cv2.imdecode(np.frombuffer(lblbuf, dtype=np.uint8), 0)
    tamper = mask > 0   # binary tampered region (handles 0/1 and 0/255 encodings)
    return image, tamper

img0, m0 = read_sample(0)
print('image', img0.shape, '| mask', m0.shape, '| tampered px', int(m0.sum()))

In [ ]:
# --- Pretrained OCR detector -> M_text (no training, per TA feedback) ---
import easyocr
from src.text_prior import OCRTextMasker

reader = easyocr.Reader(['en'], gpu=True)   # downloads its own pretrained detector weights

def easyocr_detector(image_np):
    # EasyOCR returns [(quad_points, text, conf), ...]; keep the quad polygons.
    results = reader.readtext(image_np, detail=1, paragraph=False)
    return [pts for (pts, text, conf) in results]

# `dilate` grows the text mask a few px -> higher recall. This is the main knob
# for the region-proposal high-recall target; tune it against the metrics below.
masker = OCRTextMasker(easyocr_detector, polygon=True, dilate=5)

tm0 = masker(img0)
print('text-mask area fraction on sample 0:', float((tm0 > 0).mean()))

In [ ]:
# --- Aggregate OCR recall/coverage over the split ---
from src.ocr_eval import OCRCoverageMeter
from tqdm import tqdm

meter = OCRCoverageMeter(coverage_thresh=0.5)
n = min(N_SAMPLES, num_samples)
for i in tqdm(range(n)):
    image, tamper = read_sample(i)
    if tamper.sum() == 0:
        continue                      # skip authentic images (nothing to cover)
    text_mask = masker(image)
    meter.update(text_mask, tamper)

metrics = meter.compute()
print('\n=== OCR prior vs tampered regions ===')
for k, v in metrics.items():
    print(f'{k:28s}: {v:.4f}')

In [ ]:
# --- Qualitative: where does the OCR prior MISS tampered regions? ---
import matplotlib.pyplot as plt

def overlay(image, tamper, text_mask):
    vis = image.copy()
    t = text_mask > 0
    vis[t] = (0.6 * vis[t] + 0.4 * np.array([0, 255, 0])).astype(np.uint8)  # text = green
    miss = tamper & ~t
    vis[miss] = np.array([255, 0, 0], dtype=np.uint8)   # missed tampered px = red (unrecoverable!)
    return vis

shown, i = 0, 0
plt.figure(figsize=(14, 10))
while shown < 4 and i < num_samples:
    image, tamper = read_sample(i); i += 1
    if tamper.sum() == 0:
        continue
    tmask = masker(image)
    cov = (tamper & (tmask > 0)).sum() / max(1, tamper.sum())
    plt.subplot(2, 2, shown + 1)
    plt.imshow(overlay(image, tamper, tmask))
    plt.title(f'idx {i-1}  coverage={cov:.2f}  (green=text, red=missed)')
    plt.axis('off')
    shown += 1
plt.tight_layout(); plt.show()

## Reading the results

- **`pixel_recall_coverage` / `component_recall`** — the headline numbers. As a region-proposal stage the
  prior should score high here; **red pixels** in the overlays are tampered regions it missed
  (unrecoverable downstream).
- **`text_area_ratio`** — how much of the image the prior keeps as "search space". Lower is better *at
  equal recall* (the precision / efficiency side of the trade-off).

**If recall is low:** raise `OCRTextMasker(dilate=...)`, lower EasyOCR's confidence threshold, or add
languages — then re-run. Only once recall is satisfactory does fusing the prior and training make sense
(Phase 1 in [`docs/INTEGRATION.md`](../docs/INTEGRATION.md)).

Report these numbers (quantitative) plus a couple of overlays (qualitative) in the presentation — exactly
the OCR-module evaluation the TA asked for.